# छवि उत्पादन अनुप्रयोगहरू निर्माण गर्दै (OpenAI)

LLM हरूमा केवल पाठ उत्पादन मात्र छैन। तपाईं टेक्स्ट विवरणहरूबाट चित्रहरू पनि उत्पन्न गर्न सक्नुहुन्छ। चित्रहरू एक माध्यमका रूपमा MedTech, वास्तुकला, पर्यटन, खेल विकास, मार्केटिङ, र अरू धेरै क्षेत्रमा उपयोगी छन्। यस पाठमा हामी OpenAI प्लेटफर्ममा आजका **GPT Image** मोडेलहरूसँग काम गर्छौं।

## सिकाइ लक्ष्यहरू

यो पाठको अन्त्यसम्म तपाईं सक्षम हुनुहुनेछ:

- के छवि उत्पादन हो र यो कहाँ उपयोगी हुन्छ भनेर व्याख्या गर्न।
- `gpt-image` मोडेल परिवार बुझ्न र यो विरासत DALL·E मोडेलहरू भन्दा कसरी फरक छ भने जान्न।
- छवि उत्पादन अनुप्रयोग बनाउनु र छविहरू सम्पादन गर्नु।

## छवि उत्पादन के हो?

छवि उत्पादन मोडेलहरूले पाठ प्रॉम्प्टबाट छविहरू सिर्जना गर्छन्। आधुनिक मोडेलहरू जस्तै `gpt-image` ले प्रशिक्षणको क्रममा टेक्स्ट र छविको सम्बन्ध सिक्छन्, र त्यसपछि पुनरावृत्तिभरि अनियमित आवाजलाई त्यस्तो छवि बनाउँछन् जुन तपाईंको प्रॉम्प्टसँग मेल खान्छ।

छवि मोडेलका दुई जानिन्छन् परिवारहरू:

- **`gpt-image` (OpenAI)** — यस पाठमा प्रयोग भएको हालको पुस्ता। यो टेक्स्ट-देखि-छवि उत्पादन र छवि सम्पादन (मास्कसहित इनपेन्टिङ) लाई समर्थन गर्दछ।
- **Midjourney** — यसको आफ्नै सेवा र Discord-आधारित वर्कफ्लो सहित एक लोकप्रिय तेस्रो-पक्ष मोडेल।

> पुराना OpenAI छवि मोडेलहरू — **DALL·E 2** र **DALL·E 3** — उपत्यका छन्। DALL·E 3 अब नयाँ तैनातीहरूको लागि उपलब्ध छैन, र `create_variation` जस्ता सुविधा केवल DALL·E 2 मा मात्र थिए। नयाँ अनुप्रयोगहरूको लागि `gpt-image` मोडेलहरू प्रयोग गर्नुहोस्।

> **महत्वपूर्ण:** `gpt-image` मोडेलहरूले उत्पन्न छविलाई **base64** (`b64_json`) को रूपमा फर्काउँछन्, URL को रूपमा होइन। तपाईंको कोडले base64 स्ट्रिङलाई बाइट्समा डिकोड गरेर बचत गर्छ — कुनै छवि URL डाउनलोड गर्न हुँदैन।


## तपाईंको पहिलो छवि सिर्जना अनुप्रयोग निर्माण गर्दै

त त छवि सिर्जना अनुप्रयोग बनाउन के चाहिन्छ? तपाईंलाई तलका लाइब्रेरीहरू चाहिन्छ:

- **python-dotenv**, तपाईंलाई अत्यधिक सिफारिस गरिन्छ यो लाइब्रेरी प्रयोग गर्न आफ्ना गोप्य विवरणहरू *.env* फाइलमा कोडबाट टाढा राख्न।
- **openai**, यो लाइब्रेरी हो जसले तपाईंलाई OpenAI API सँग अन्तरक्रिया गर्न प्रयोग गर्नुहुनेछ।
- **pillow**, Python मा छविहरूसँग काम गर्न।


1. निम्न सामग्री भएको *.env* फाइल सिर्जना गर्नुहोस्:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. माथि उल्लेख गरिएका पुस्तकालयहरू *requirements.txt* नामक फाइलमा यसरी संकलन गर्नुहोस्:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. अब, भर्चुअल वातावरण सिर्जना गरी पुस्तकालयहरू स्थापना गर्नुहोस्:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Windows को लागि, आफ्नो भर्चुअल वातावरण सिर्जना गर्न र सक्रिय गर्न तलका आदेशहरू प्रयोग गर्नुहोस्:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. *app.py* नामक फाइलमा निम्न कोड थप्नुहोस्:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # dotenv आयात गर्नुहोस्
    dotenv.load_dotenv()

    # OpenAI वस्तु सिर्जना गर्नुहोस् (.env बाट OPENAI_API_KEY पढ्छ)
    client = openai.OpenAI()


    try:
        # छवि उत्पादन API प्रयोग गरी छवि सिर्जना गर्नुहोस्
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # यहाँ आफ्नो प्रॉम्प्ट टेक्स्ट प्रविष्ट गर्नुहोस्
            size='1024x1024',
            n=1
        )
        # संग्रह गरिएको छविका लागि डाइरेक्टरी सेट गर्नुहोस्
        image_dir = os.path.join(os.curdir, 'images')

        # यदि डाइरेक्टरी अवस्थित छैन भने, यसलाई सिर्जना गर्नुहोस्
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # छवि पथ आरम्भ गर्नुहोस् (नोट: फाइल प्रकार png हुनुपर्छ)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image मोडलहरूले छविलाई URL होइन base64 (b64_json) मा फर्काउँछन्
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # डिफल्ट छवि दृश्यकर्तामा छवि प्रदर्शन गर्नुहोस्
        image = Image.open(image_path)
        image.show()

    # अपवादहरू समात्नुहोस्
    except openai.BadRequestError as err:
        print(err)

    ```

यस कोडलाई व्याख्या गरौं:

- पहिले, हामीले आवश्यक पर्ने पुस्तकालयहरू आयात गर्छौं, जसमा OpenAI पुस्तकालय, dotenv पुस्तकालय, base64 मोड्युल, र Pillow पुस्तकालय समावेश छन्।

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- त्यसपछि, हामी क्लाइन्ट सिर्जना गर्छौं, जुन तपाईंको ``.env`` बाट API कुञ्जी पढ्छ।

    ```python
    # OpenAI वस्तु सिर्जना गर्नुहोस्
    client = openai.OpenAI()
    ```

- अर्को, हामी चित्र सिर्जना गर्छौं:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # यहाँ आफ्नो प्रॉम्प्ट टेक्स्ट प्रविष्ट गर्नुहोस्
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` मोडेलहरूले चित्रलाई `data[0].b64_json` मा **base64** स्ट्रिङको रूपमा फिर्ता गर्छन्। हामी यसलाई बाइटहरूमा डिकोड गर्छौं र फाइलमा लेख्छौं—डाउनलोड गर्न URL हुँदैन।

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- अन्तमा, हामी चित्र खोल्छौं र चित्र हेर्ने मानक उपकरण प्रयोग गरी देखाउँछौं:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### चित्र सिर्जना सम्बन्धी थप विवरण

`images.generate` का प्यारामिटरहरू हेरौं:

- **model**, प्रयोग गर्नुपर्ने चित्र मोडेल हो (जस्तै `gpt-image-1`)।
- **prompt**, चित्र सिर्जना गर्न प्रयोग गरिएको पाठ प्रॉम्प्ट हो। यहाँ "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils" छ।
- **size**, सिर्जित चित्रको आकार हो (जस्तै `1024x1024`, `1536x1024`, `1024x1536`, वा `"auto"`)।
- **n**, सिर्जना गरिएका चित्रहरूको संख्या हो। यहाँ हामीले एउटा सिर्जना गरेका छौं।

> चित्र मोडेलहरूले `temperature` प्यारामिटर लिँदैनन् — त्यो पाठ-निर्माण नियन्त्रण हो। विविधता पाउनका लागि API पुनः कल गर्नुहोस्; विविधता घटाउन प्रॉम्प्टलाई थप विशेष बनाउनुहोस्।

## चित्र सिर्जनाका थप क्षमता

तपाईंले केही पंक्तिहरूको Python कोडले चित्र कसरी सिर्जना गर्ने देख्नुभयो। `gpt-image` मोडेलहरूले एउटा अवस्थित चित्रलाई पनि **सम्पादन** गर्न सक्छन्। एउटा चित्र, वैकल्पिक **मास्क** (जसले परिवर्तन गर्ने क्षेत्र चिन्न्छ), र प्रॉम्प्ट प्रदान गरेर, तपाईं चित्रको भाग परिवर्तित गर्न सक्नुहुन्छ—जस्तै हाम्रो खरायोलाई टोपी थप्न।

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# सम्पादनहरू पनि base64 को रूपमा फिर्ता गरिन्छन्
edited_image = base64.b64decode(response.data[0].b64_json)
```

आधार चित्रमा केवल खरायो छ; अन्तिम चित्रमा खरायोमा टोपी छ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
